# Convert .mp3 to .wav

In [1]:
import os
from pydub import AudioSegment

root_dir = "/content/drive/MyDrive/Pipeline_Evaluation"
speakers = ["speaker_01", "speaker_02", "speaker_03"]

for spk in speakers:
  spk_dir = os.path.join(root_dir, spk)
  for category in ["statements", "questions", "commands", "complex", "named_entities"]:
    cat_dir = os.path.join(spk_dir, category)
    for file in os.listdir(cat_dir):
      if file.endswith(".mp3"):
        mp3_path = os.path.join(cat_dir, file)
        wav_path = os.path.splitext(mp3_path)[0] + ".wav"

        # Convert to wav 16KHz mono
        audio = AudioSegment.from_file(mp3_path, format="mp3")
        audio = audio.set_frame_rate(16000).set_channels(1)
        audio.export(wav_path, format="wav")

        # Delete mp3 file (Optional)
        # os.remove(mp3_path)


In [3]:
import os

root_dir = "/content/drive/MyDrive/Pipeline_Evaluation"

deleted_files = []

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.lower().endswith(".mp3"):
            file_path = os.path.join(root, file)
            os.remove(file_path)
            deleted_files.append(file_path)

print(f"Deleted {len(deleted_files)} mp3 files.")


Deleted 900 mp3 files.


# Building Metadata

In [5]:
import pandas as pd
import os

root_dir = "/content/drive/MyDrive/Pipeline_Evaluation"
csv_path = os.path.join(root_dir, "Dataset_For_Evaluation.csv")

# Load source translations
df = pd.read_csv(csv_path)

# Expected columns:
# Statements, Translation, Questions, Translation, Command, Translation, Complex, Translation, Named Entities/Numbers, Translation

records = []

for spk in ["speaker_01", "speaker_02", "speaker_03"]:
    for category, nep_col, eng_col in [
        ("statements", "Statements", "Translation"),
        ("questions", "Questions", "Translation.1"),
        ("commands", "Command", "Translation.2"),
        ("complex", "Complex", "Translation.3"),
        ("named_entities", "Named Entities/Numbers", "Translation.4"),
    ]:
        cat_dir = os.path.join(root_dir, spk, category)
        if not os.path.exists(cat_dir):
            continue
        files = sorted([f for f in os.listdir(cat_dir) if f.endswith(".wav")], key=lambda x: int(x.split(".")[0]))

        for i, file in enumerate(files):
            nep = df[nep_col].iloc[i]
            eng = df[eng_col].iloc[i]

            records.append({
                "id": f"{spk}_{category}_{i+1}",
                "speaker_id": spk,
                "category": category,
                "audio_path": f"{spk}/{category}/{file}",
                "source_text": str(nep),
                "reference_translation": str(eng)
            })

# Save master dataset
dataset_df = pd.DataFrame(records)
dataset_df.to_csv(os.path.join(root_dir, "metadata.csv"), index=False)

print(dataset_df.head())


                        id  speaker_id    category  \
0  speaker_01_statements_1  speaker_01  statements   
1  speaker_01_statements_2  speaker_01  statements   
2  speaker_01_statements_3  speaker_01  statements   
3  speaker_01_statements_4  speaker_01  statements   
4  speaker_01_statements_5  speaker_01  statements   

                    audio_path       source_text  reference_translation  
0  speaker_01/statements/1.wav      यो किताब हो।        This is a book.  
1  speaker_01/statements/2.wav  ऊ विद्यार्थी हो।       He is a student.  
2  speaker_01/statements/3.wav     म नेपाली हुँ।           I am Nepali.  
3  speaker_01/statements/4.wav      पानी चिसो छ।     The water is cold.  
4  speaker_01/statements/5.wav      खाना मिठो छ।  The food is delicious  


In [7]:
dataset_df.head(900)

,id,speaker_id,category,audio_path,source_text,reference_translation
0,speaker_01_statements_1,speaker_01,statements,speaker_01/statements/1.wav,यो किताब हो।,This is a book.
1,speaker_01_statements_2,speaker_01,statements,speaker_01/statements/2.wav,ऊ विद्यार्थी हो।,He is a student.
2,speaker_01_statements_3,speaker_01,statements,speaker_01/statements/3.wav,म नेपाली हुँ।,I am Nepali.
3,speaker_01_statements_4,speaker_01,statements,speaker_01/statements/4.wav,पानी चिसो छ।,The water is cold.
4,speaker_01_statements_5,speaker_01,statements,speaker_01/statements/5.wav,खाना मिठो छ।,The food is delicious
...,...,...,...,...,...,...
895,speaker_03_named_entities_26,speaker_03,named_entities,speaker_03/named_entities/26.wav,ओलम्पिक खेल,The Olympic Games
896,speaker_03_named_entities_27,speaker_03,named_entities,speaker_03/named_entities/27.wav,महात्मा गान्धी,Mahatma Gandhi
897,speaker_03_named_entities_28,speaker_03,named_entities,speaker_03/named_entities/28.wav,अल्बर्ट आइन्स्टाइन,Albert Einstein
898,speaker_03_named_entities_29,speaker_03,named_entities,speaker_03/named_entities/29.wav,भर्साइलको सन्धि,The Treaty of Versailles


In [8]:
len(dataset_df)

900

In [11]:
# !pip install huggingface_hub -q

In [12]:
from huggingface_hub import notebook_login
notebook_login()

In [13]:
!pwd

/content


In [3]:
%cd /content/drive/MyDrive/Pipeline_Evaluation

/content/drive/MyDrive/Pipeline_Evaluation


In [2]:
!pwd

/content


In [21]:
!pip install torch torchaudio torchcodec --q

In [4]:
from datasets import load_dataset, Audio

# Load local dataset
dataset = load_dataset("csv", data_files="/content/drive/MyDrive/Pipeline_Evaluation/metadata.csv")

# Rename split to evaluation
from datasets import DatasetDict
dataset = DatasetDict({"evaluation": dataset["train"]})

# Cast audio
dataset = dataset.cast_column("audio_path", Audio(sampling_rate=16_000))

# Test
example = dataset["evaluation"][0]
print(example.keys())
print(example["audio_path"])


dict_keys(['id', 'speaker_id', 'category', 'audio_path', 'source_text', 'reference_translation'])


In [5]:
print(dataset)

DatasetDict({
    evaluation: Dataset({
        features: ['id', 'speaker_id', 'category', 'audio_path', 'source_text', 'reference_translation'],
        num_rows: 900
    })
})


In [6]:
dataset.push_to_hub("iamTangsang/nepali_to_english_pipeline_evaluation")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   1%|          |  611kB / 70.6MB            

CommitInfo(commit_url='https://huggingface.co/datasets/iamTangsang/nepali_to_english_pipeline_evaluation/commit/af22f9a77fdb12b995a733314fa7ee8ecf04db09', commit_message='Upload dataset', commit_description='', oid='af22f9a77fdb12b995a733314fa7ee8ecf04db09', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/iamTangsang/nepali_to_english_pipeline_evaluation', endpoint='https://huggingface.co', repo_type='dataset', repo_id='iamTangsang/nepali_to_english_pipeline_evaluation'), pr_revision=None, pr_num=None)

In [7]:
from datasets import load_dataset, Audio

ds = load_dataset("iamTangsang/nepali_to_english_pipeline_evaluation", split="evaluation")
ds = ds.cast_column("audio_path", Audio(sampling_rate=16_000))

print(ds[0])

README.md:   0%|          | 0.00/533 [00:00<?, ?B/s]

data/evaluation-00000-of-00001.parquet:   0%|          | 0.00/70.6M [00:00<?, ?B/s]

Generating evaluation split:   0%|          | 0/900 [00:00<?, ? examples/s]

{'id': 'speaker_01_statements_1', 'speaker_id': 'speaker_01', 'category': 'statements', 'audio_path': <datasets.features._torchcodec.AudioDecoder object at 0x7821cfadb920>, 'source_text': 'यो किताब हो।', 'reference_translation': 'This is a book.'}


In [8]:
print(ds)

Dataset({
    features: ['id', 'speaker_id', 'category', 'audio_path', 'source_text', 'reference_translation'],
    num_rows: 900
})
